# 2 · Training, model selection and the freeze

Every head is trained on the cached features; the backbone never changes.

* For each head, the learning rate in {3e-4, 1e-3, 3e-3} with the best **validation** top-1 on seed 0 is
  kept (ties: lower validation NLL; `experiments/<exp>/lr_selection.json`). The ablation, open-set and
  token-width runs borrow the rate selected for their head in `tsinghua_main` (`lr_selection_from`, `lr_like`)
  and select nothing themselves; each other backbone has its own grid.
* Seeds 0, 1 and 2 are then trained at that rate, for at most 30 epochs, with early stopping on validation
  top-1 (patience 8).
* No cell in this notebook reads a test split. It ends with the selection freeze (`scripts/freeze.py`).

Heads, at equal parameter counts for all re-rankers (302,211 on Tsinghua Dogs):

| Group | Heads |
|---|---|
| Baselines | `linear`, `attnprobe`, `classattn_all` |
| Controls (same size, rival-blind queries) | `rr_class`, `rr_center`, `rr_random` |
| Proposed (rival-conditioned queries) | `rr_rival`, `rr_rival_soft`, `rr_pairwise` |

## Setup (run at the start of every session)

Runtime → Change runtime type → **T4 GPU** (any CUDA GPU works; the study uses a free-tier T4, 15 GB).
Mounting Google Drive is optional but strongly advised: `scripts/colab_sync.sh` mirrors everything under
`/content/work` (manifests, audits, runs, freezes, results, logs, queue state) to Drive, so a recycled runtime
loses only the feature caches. Notebook 1 rebuilds those from the images. `push()` (below) first copies every
token PCA (`pca.npz`) into `/content/work/pca_backup`, and this cell puts it back, so rebuilt caches use the
basis the heads were trained in (`extract_features.py` reuses an existing `pca.npz` instead of fitting a new one).

**Long jobs run detached.** A cell `!nohup cmd &` stays busy, and keeps the kernel busy, until the job ends,
because the job holds the cell's terminal. Every long job here is started as
`!setsid nohup bash -c '…' > /content/work/logs/<name>.log 2>&1 < /dev/null &`: the cell returns at once, and a
watch cell reads the log. The same commands can be typed into Colab's Terminal panel (without the `!`) after
exporting the variables `scripts/colab_env.py` sets, since a terminal does not have the kernel's environment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # optional: skip this cell to keep everything on the runtime's disk

In [ ]:
# Where the code comes from: a git URL, or a zip of this repository (top folder DogBreed-PhD-Research)
REPO_URL = ""                                                   # e.g. the URL of your fork
REPO_ZIP = "/content/drive/MyDrive/DogBreed-PhD-Research.zip"
import os, subprocess
D = "/content/DogBreed-PhD-Research"                            # the queue files cd here: keep this path
if not os.path.isdir(D):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, D], check=True)
    elif os.path.isfile(REPO_ZIP):
        subprocess.run(["unzip", "-q", REPO_ZIP, "-d", "/content"], check=True)
    else:
        raise SystemExit("set REPO_URL, or put the repository zip at REPO_ZIP")
os.chdir(D)
exec(open("scripts/colab_env.py").read())                     # paths: caches on local disk, the rest on Drive
!pip install -q --no-deps -e . && python -m pytest -q tests 2>&1 | tail -2
!bash scripts/colab_sync.sh pull                                # restore earlier sessions' work, if any
!rsync -a /content/work/pca_backup/ /content/features/ 2>/dev/null && find /content/features -name pca.npz
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


def push():
    """Copy every token PCA of the caches into /content/work/pca_backup, then mirror /content/work to Drive."""
    !rsync -am --include='*/' --include='pca.npz' --include='pca.json' --exclude='*' /content/features/ /content/work/pca_backup/
    !bash scripts/colab_sync.sh push

## Helpers

In [ ]:
import glob, json
import yaml
E, R, W = os.environ["DOGBREED_EXPERIMENTS"], os.environ["DOGBREED_RESULTS"], os.environ["WORK"]
EXPS = ["tsinghua_main", "stanford_main", "tsinghua_ablate", "tsinghua_openset", "tsinghua_resnet50",
        "tsinghua_convnext_tiny", "tsinghua_clip_vitb16", "tsinghua_tok32", "tsinghua_tok64"]


def complete(exp):
    """Every run of the experiment's config has finished for every seed: best.pt and summary.json, the two
    files scripts/freeze.py reads (train_heads.py writes best.pt just before summary.json)."""
    cfg = yaml.safe_load(open(f"configs/experiments/{exp}.yaml"))
    return all(os.path.exists(f"{E}/{exp}/{r['tag']}_s{s}/{f}") for r in cfg["runs"] for s in cfg["seeds"]
               for f in ("best.pt", "summary.json"))


def frozen(exp):
    """The experiment's freeze.json (scripts/freeze.py), or None."""
    f = f"{E}/{exp}/freeze.json"
    return json.load(open(f)) if os.path.exists(f) else None


def final_done(exp):
    """scripts/final_all.sh finished this experiment (its log ends with FINAL DONE)."""
    f = f"{W}/logs/final_{exp}.log"
    return os.path.exists(f) and f"FINAL DONE {exp}" in open(f).read()


def running(pattern):
    """A process whose command line matches (pgrep -f); write "[x]yz" so a shell never matches itself."""
    return subprocess.run(["pgrep", "-f", pattern], capture_output=True).returncode == 0

## The main experiment: `tsinghua_main` (about 1.5–3 h on a T4)

`scripts/run_experiment.sh` runs the learning-rate grid on seed 0 (`run_heads.py --select-lr`), then seeds 0–2
at the chosen rates (`run_heads.py`). Finished runs are skipped; an interrupted run resumes from its `last.pt`.
The log ends with `TRAINING DONE configs/experiments/tsinghua_main.yaml`. Start it exactly as below: queue A
recognises it by this command line.

In [ ]:
if running("[r]un_experiment.sh configs/experiments/tsinghua_main.yaml"):
    print("tsinghua_main is already running")
else:
    !setsid nohup bash -c 'bash scripts/run_experiment.sh configs/experiments/tsinghua_main.yaml' > /content/work/logs/tsinghua_main.log 2>&1 < /dev/null &

## Everything else, unattended: queues A, B and C

`scripts/queue.sh` runs the `name | command` lines of a queue file in order, each once. A finished job leaves
`<name>.done`, a failed one `<name>.failed` (and the queue moves on), in `/content/work/logs/queue_<a|b|c>/`; its
output goes to `/content/work/logs/<name>.log`. Lines added while the queue runs are picked up; the queue exits
once every job is marked and the file has an `END` line.

| Queue | Jobs, in order |
|---|---|
| A, `configs/queues/a.txt`: Tsinghua-sized jobs (about 9 GB of tokens on the GPU each), one at a time | `wait_main` (until `tsinghua_main` has finished) → `ablate` (`tsinghua_ablate`) → `openset_split` (`make_openset_split.py --dataset tsinghua --unknown 30 --seed 0`) → `openset` (`tsinghua_openset`) → `wait_bb` (until queue B's `extract_clip_vitb16` is marked) → `tok32`, `tok64` |
| B, `configs/queues/b.txt`: Stanford-sized jobs, alongside A | `wait_stanford` (until `pipeline_stanford.sh` has finished) → `stanford_main` → `extract_resnet50`, `extract_convnext_tiny`, `extract_clip_vitb16` (Tsinghua caches, orig and hflip) |
| C, `configs/queues/c.txt`: the other backbones, alongside A | `wait_b` (until queue B's `extract_clip_vitb16` is marked) → `bb_resnet50`, `bb_convnext_tiny`, `bb_clip_vitb16` (each with its own learning-rate grid) |

Start the queues only once `tsinghua_main` is running (or complete): otherwise `wait_main` passes at once and
the ablations fail for want of its learning rates. A queue that is already running is not started twice.

In [ ]:
assert running("[r]un_experiment.sh configs/experiments/tsinghua_main.yaml") or complete("tsinghua_main"), \
    "start tsinghua_main first (cell above)"
for q in ("a", "b", "c"):
    if running(f"[q]ueue.sh configs/queues/{q}.txt"):
        print(f"queue {q} is already running")
    else:
        !setsid nohup bash -c 'bash scripts/queue.sh configs/queues/{q}.txt' > /content/work/logs/queue_{q}.log 2>&1 < /dev/null &

## Watch

In [ ]:
!tail -n 2 /content/work/logs/tsinghua_main.log /content/work/logs/queue_a.log /content/work/logs/queue_b.log /content/work/logs/queue_c.log
!ls /content/work/logs/queue_a /content/work/logs/queue_b /content/work/logs/queue_c 2>/dev/null; nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader
push()

A failed job's output is in `/content/work/logs/<job>.log`. To run it again, delete its `.failed` marker: a
running queue picks it up, a queue that has exited must be started again (cell above).

## After a recycled runtime

Run the setup cells and the last section of notebook 1 (the caches), then the two launch cells above again:
finished runs are skipped and jobs marked done are not repeated. A run cut off mid-way starts again from its
first epoch, since `last.pt` is not mirrored to Drive. The open-set split is written into the repository
checkout (`configs/openset/`), which is not mirrored either; write it again (same seed, same breeds) before the
open-set runs resume:

In [ ]:
![ -f configs/openset/tsinghua_unknown30_s0.json ] || python -I scripts/make_openset_split.py --dataset tsinghua --unknown 30 --seed 0

## Validation results (no test data)

In [ ]:
import pandas as pd
for f in sorted(glob.glob(f"{E}/*/lr_selection.json")):
    print(f.split("/")[-2], json.load(open(f)))
rows = []
for f in sorted(glob.glob(f"{E}/*/*_s[0-9]/summary.json")):
    exp, run = f.split("/")[-3:-1]
    if "_lr" in run:                                   # learning-rate grid runs
        continue
    r = json.load(open(f))
    rows.append({"experiment": exp, "run": run, "lr": r["config"].get("lr"), "val_top1": r["val_top1"],
                 "val_nll": r["val_nll"], "best_epoch": r["best_epoch"], "epochs_run": r["epochs_run"],
                 "params": r["params"]})
print(pd.DataFrame(rows).to_string(index=False))

## The selection freeze — stop here

When every configured run of an experiment has finished, `scripts/freeze.py --experiment <exp>` writes
`experiments/<exp>/freeze.json`: the time (UTC), each head's learning rate, and for every seed run its
validation metrics, best epoch and the SHA-256 of its `best.pt`. Every script that reads a test split refuses
an experiment without it, a run it does not list, and a checkpoint whose hash has changed since
(`require_freeze` in `src/dogbreed/evaluation/runs.py`). `freeze.py` refuses a head with fewer than
`--expect-seeds` (3) finished seeds, and a second freeze without `--force` (a protocol event that belongs in the
research log). It only sees heads with at least one finished seed, so this cell freezes an experiment only when
`complete()` finds every run of its config.

Paste each printed summary, with its time, into the section "Model selection freeze" of
`research_log/research_log.md`. Only then open notebook 3 for that experiment.

In [ ]:
for exp in EXPS:
    fz = frozen(exp)
    if fz:
        print(f"{exp}: frozen {fz['frozen_at_utc']} UTC, {len(fz['runs'])} runs")
    elif not complete(exp):
        print(f"{exp}: training not complete, not frozen")
    else:
        !python -I scripts/freeze.py --experiment {exp}
push()